# Reproducible data + ML pipeline on StudyChat · Pipeline reproducible de datos y ML sobre StudyChat
**Deliverable 05 (v2)** · Research Methods and Scientific Integrity · UNMSM 2026-II · Deisy Lizbeth Acosta Ticse

**Question · Pregunta:** can we detect, from the text alone, when a student asks a generative-AI assistant to *do the work* instead of helping them understand?
¿Podemos detectar, solo por el texto, cuándo un estudiante le pide a la IA que *haga el trabajo* en lugar de ayudarle a entender?

**Why it matters for the thesis · Por qué importa para la tesis:** the thesis agent must recognise a delegation request before it can scaffold reflection. This notebook prototypes that detector on real, public student–ChatGPT dialogues.

## Step 0 · Setup · Preparación
**What / Qué:** download the repository and install the *exact* library versions.
**Why / Por qué:** different library versions can produce different numbers; pinning them is the first condition of reproducibility.
If Colab reports that NumPy changed, use *Runtime → Restart session* and run the cells below Step 0 again.

In [ ]:
!git clone https://github.com/hjdeysi/studychat-repro-pipeline.git
%cd studychat-repro-pipeline
!pip install -q -r requirements.txt
!python --version

## Step 1 · Code history (Git) · Historial del código
**Why / Por qué:** small, labelled commits let anyone see *how* the pipeline was built and return to any earlier state. Each MLflow run stores the commit that produced it.

In [ ]:
!git log --oneline

## Step 2 · Data lake + fingerprint · Data lake y huella de los datos
**What is a data lake? / ¿Qué es un data lake?** A store that keeps data in *zones* by level of processing, so the original is never overwritten:

| Zone · Zona | Content · Contenido | In Git? |
|---|---|---|
| `raw` | the source file exactly as published (immutable) · el archivo tal como se publicó | No (626 MB) |
| `interim` | rows that passed validation · filas que pasaron la validación | No (regenerable) |
| `processed` | model-ready table · tabla lista para el modelo | **Yes** |

**Fingerprint / Huella:** a SHA-256 checksum. If one byte of the data changes, the checksum changes, so `DATA_MANIFEST.json` proves which data produced the results.

Two options · Dos opciones:
- **A (full):** download the raw file from Hugging Face at a pinned revision and rebuild every zone (~626 MB, a few minutes).
- **B (quick):** start from the committed processed table. Run only one of the two cells below.

In [ ]:
# Option A · full pipeline from the source (download + validate + transform)
!PYTHONHASHSEED=0 python -m src.ingest
!PYTHONHASHSEED=0 python -m src.validate
!PYTHONHASHSEED=0 python -m src.transform

In [ ]:
# Option B · skip raw: verify the committed processed table against the manifest
import json, hashlib
m = json.load(open("data/DATA_MANIFEST.json"))
h = hashlib.sha256(open(m["processed"]["file"], "rb").read()).hexdigest()
print("manifest:", m["processed"]["sha256"]); print("file    :", h); print("MATCH" if h == m["processed"]["sha256"] else "MISMATCH")

## Step 3 · Data-quality audit · Auditoría de calidad
**Why / Por qué:** a model is only as trustworthy as its data. The validation stage drops rows that break the schema and *counts* them, instead of fixing them silently. It also confirms that no student appears in both cohorts.

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open("reports/data_quality.json")), indent=2))
df = pd.read_csv("data/processed/studychat_prompts.csv.gz")
df.groupby("split").agg(turns=("prompt", "size"), students=("user_id", "nunique"), delegation_rate=("y_delegation", "mean"))

## Step 4 · The pipeline · El pipeline (`src/train.py`)
**Target / Objetivo:** `y_delegation = 1` if the turn asks the assistant to write or edit code/text.
**No leakage / Sin fuga de datos:** train on Fall 2024, test once on Spring 2025 (different students). TF-IDF and the classifier sit inside one scikit-learn `Pipeline`, so in each cross-validation fold they learn only from the training folds. Folds are grouped by student, so one student's turns never appear on both sides.
**Baseline / Línea base:** always predicting the majority class. A model is only useful if it clearly beats it.

In [ ]:
df[["topic", "label_fine", "y_delegation", "prompt"]].sample(5, random_state=0)

## Step 5 · Same seed, same result · Misma semilla, mismo resultado
**Why / Por qué:** randomness (fold shuffling, solver order) is controlled by a seed. Running the same seed twice must give identical numbers.

In [ ]:
!PYTHONHASHSEED=0 python -m src.train --seed 42 > run1.txt
!PYTHONHASHSEED=0 python -m src.train --seed 42 > run2.txt
!diff run1.txt run2.txt && echo "IDENTICAL" ; tail -1 run1.txt

## Step 6 · Record every run (MLflow) · Registrar cada ejecución
**Why / Por qué:** one seed can be lucky. Five seeds show how stable the result is; MLflow records parameters, data checksum, Hugging Face revision and git commit of every run, so each number can be traced.

In [ ]:
!PYTHONHASHSEED=0 python -m src.run_seeds 2>&1 | grep -E "^\[seed|Selected"
pd.read_csv("reports/results_summary.csv")[["model", "cv_f1_macro_mean", "test_f1_macro_mean", "test_f1_macro_std", "test_roc_auc_mean", "test_unseen_f1_macro_mean"]]

**Expected / Esperado:** selected model (TF-IDF + logistic regression) test macro-F1 **0.7607 ± 0.0022** across seeds 0, 1, 2, 3, 42; majority baseline 0.4138.
To browse runs locally: `mlflow ui --backend-store-uri sqlite:///mlflow.db`.

## Step 7 · A recipe for a stranger · Receta para un extraño
```bash
git clone https://github.com/hjdeysi/studychat-repro-pipeline.git && cd studychat-repro-pipeline
pip install -r requirements.txt
PYTHONHASHSEED=0 python -m src.pipeline                   # full, from Hugging Face
PYTHONHASHSEED=0 python -m src.pipeline --from-processed  # quick, from committed table
```

## ✅ Self-check · Autoevaluación
Checks checksums, cohort separation and outputs. All lines must say PASS.

In [ ]:
!python -m src.checks

## Limitations · Limitaciones
1. Labels were generated by GPT-4.1, so scores measure agreement with GPT-4.1, not with human experts.
2. "Delegation" is an operational proxy; it does not measure intent or virtue.
3. Cross-validation (0.81) overestimates performance on a new cohort (0.76).
4. One U.S. course in English; the thesis will collect its own Spanish-language corpus.

**Data:** McNichols, Ikram & Lan (2026), *The StudyChat Dataset*, LAK26, 53–63 · CC BY 4.0.
**AI-use disclosure:** Claude (Anthropic) assisted with code and documentation; the author reviewed and verified all results.